# PR-2: Robust Regression Engine

## Part-A: Conceptual Foundation (Theory)

1. What is Regularization in Machine Learning? Why is it needed?
- Regularization in machine learning is a technique used to prevent overfitting by adding a penalty term to the loss function. It helps to constrain the model complexity, ensuring that the model generalizes well to unseen data. Regularization is needed because complex models can fit the training data too closely, capturing noise and leading to poor performance on new data. By penalizing large coefficients, regularization encourages simpler models that are more robust and less likely to overfit.

2. Difference between Ridge Regression (L2) and Lasso Regression (L1)
- Ridge Regression (L2) adds a penalty term proportional to the square of the magnitude of the coefficients to the loss function. It shrinks the coefficients towards zero but does not set them exactly to zero, allowing all features to contribute to the model. Lasso Regression (L1), on the other hand, adds a penalty term proportional to the absolute value of the coefficients. This can lead to some coefficients being exactly zero, effectively performing feature selection by excluding less important features from the model.


3. What is Cross-Validation and why is it important?
- Cross-validation is a statistical technique used to evaluate the performance of a machine learning model by partitioning the data into subsets, training the model on some subsets (training set) and validating it on the remaining subsets (validation set). It is important because it provides a more reliable estimate of model performance, helps in tuning hyperparameters, and reduces the risk of overfitting by ensuring that the model generalizes well to unseen data. Common methods include k-fold cross-validation and leave-one-out cross-validation.


4. Explain the following cross-validation techniques:
   - a. K-Fold Cross-Validation
     - In K-Fold Cross-Validation, the dataset is divided into 'K' equal-sized folds. The model is trained on K-1 folds and validated on the remaining fold. This process is repeated K times, with each fold serving as the validation set once. The final performance metric is averaged over all K iterations, providing a robust estimate of model performance.
   
   - b. Stratified K-Fold Cross-Validation
     - Stratified K-Fold Cross-Validation is a variation of K-Fold Cross-Validation that ensures each fold has a similar distribution of the target variable, especially in cases of imbalanced datasets. This technique helps maintain the representativeness of each fold, leading to more reliable performance estimates and better generalization of the model.

   - c. Leave-One-Out Cross-Validation (LOOCV)
     - Leave-One-Out Cross-Validation is a special case of K-Fold Cross-Validation where K equals the number of data points in the dataset. Each data point is used as a single validation instance while the model is trained on all other data points. This method provides an unbiased estimate of model performance but can be computationally expensive for large datasets.

   - d. Time Series Split
     - Time Series Split is a cross-validation technique specifically designed for time series data. It respects the temporal order of the data by creating training and validation sets that maintain the chronological sequence. The model is trained on past data and validated on future data, ensuring that the evaluation reflects real-world scenarios where future data is not available during training. This method helps prevent data leakage and provides a more accurate assessment of model performance in time-dependent contexts.

5. Why are tree-based models less sensitive to feature scaling ?
- Tree-based models, such as decision trees, random forests, and gradient boosting machines, are less sensitive to feature scaling because they split the data based on feature values rather than relying on distance metrics. These models make decisions by evaluating thresholds for each feature, which means that the absolute scale of the features does not affect the splitting criteria. As a result, tree-based models can handle features with different scales without requiring normalization or standardization, making them robust to variations in feature magnitudes.

## Part-B: Dataset Understanding & Preparation

In [33]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import TimeSeriesSplit
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error


In [34]:
df = pd.read_csv("PR-2.csv")
df.head()

,property_id,sale_date,area_sqft,bedrooms,bathrooms,location_score,property_age,distance_city_km,near_school,near_metro,crime_rate_index,house_price_inr
0,200001,2014-01-01,2181,6,4,8.1,21,3.8,0,0,4.84,35154898
1,200002,2019-12-01,2383,5,4,5.3,28,10.9,1,1,2.89,26710893
2,200003,2016-10-01,1047,3,3,5.9,7,27.5,0,1,4.04,11216242
3,200004,2013-03-01,1753,4,3,7.0,27,12.1,0,0,3.28,21984310
4,200005,2013-07-01,1728,4,4,10.0,32,1.4,0,1,3.84,25080429


In [35]:
df['sale_date'] = pd.to_datetime(df['sale_date'])

df['sale_year'] = df['sale_date'].dt.year
df['sale_month'] = df['sale_date'].dt.month
df['sale_date'] = df['sale_date'].dt.date

df.head()

,property_id,sale_date,area_sqft,bedrooms,bathrooms,location_score,property_age,distance_city_km,near_school,near_metro,crime_rate_index,house_price_inr,sale_year,sale_month
0,200001,2014-01-01,2181,6,4,8.1,21,3.8,0,0,4.84,35154898,2014,1
1,200002,2019-12-01,2383,5,4,5.3,28,10.9,1,1,2.89,26710893,2019,12
2,200003,2016-10-01,1047,3,3,5.9,7,27.5,0,1,4.04,11216242,2016,10
3,200004,2013-03-01,1753,4,3,7.0,27,12.1,0,0,3.28,21984310,2013,3
4,200005,2013-07-01,1728,4,4,10.0,32,1.4,0,1,3.84,25080429,2013,7


### 6. Identify features and target variable.

In [36]:
X = df.drop(columns=['property_id', 'sale_date', 'house_price_inr'])

y = df['house_price_inr']

print("Features:")
print(X.head())

print("\nTarget Variable:")
print(y.head())

Features:
   area_sqft  bedrooms  bathrooms  location_score  property_age  \
0       2181         6          4             8.1            21   
1       2383         5          4             5.3            28   
2       1047         3          3             5.9             7   
3       1753         4          3             7.0            27   
4       1728         4          4            10.0            32   

   distance_city_km  near_school  near_metro  crime_rate_index  sale_year  \
0               3.8            0           0              4.84       2014   
1              10.9            1           1              2.89       2019   
2              27.5            0           1              4.04       2016   
3              12.1            0           0              3.28       2013   
4               1.4            0           1              3.84       2013   

   sale_month  
0           1  
1          12  
2          10  
3           3  
4           7  

Target Variable:
0    35154

### 7. Perform a train-test split.

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

Training Data: (3040, 11)
Testing Data: (760, 11)


### 8. Apply basic preprocessing (scaling where required).

In [38]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training data shape:", X_train_scaled.shape)
print("Testing data shape:", X_test_scaled.shape)

Training data shape: (3040, 11)
Testing data shape: (760, 11)


## Part-C: Regularized Linear Models

### 9. Implement Ridge Regression (L2).

In [39]:
ridge_model = Ridge(alpha=1.0)

ridge_model.fit(X_train_scaled, y_train)

y_pred_ridge = ridge_model.predict(X_test_scaled)

print("Predicted House Prices:")
print(y_pred_ridge[:5])

Predicted House Prices:
[10797094.22839241 25656018.3790172  23084282.57602871 21706686.34519236
 30857286.89144886]


### 10. Implement Lasso Regression (L1).

In [40]:
lasso_model = Lasso(alpha=1.0)

lasso_model.fit(X_train_scaled, y_train)

y_pred_lasso = lasso_model.predict(X_test_scaled)

print("Predicted House Prices:")
print(y_pred_lasso[:5])

Predicted House Prices:
[10796423.8970383  25651408.85714514 23085975.1412157  21709671.68771617
 30857686.40116063]


### 11. Tune the regularization parameter (alpha) using cross-validation.

In [41]:
alpha_values = {'alpha': [0.01, 0.1, 1, 10, 100]}

ridge_cv = GridSearchCV(Ridge(), alpha_values, cv=5)
ridge_cv.fit(X_train_scaled, y_train)

print("Best Alpha for Ridge:", ridge_cv.best_params_['alpha'])

lasso_cv = GridSearchCV(Lasso(max_iter=10000), alpha_values, cv=5)
lasso_cv.fit(X_train_scaled, y_train)

print("Best Alpha for Lasso:", lasso_cv.best_params_['alpha'])

Best Alpha for Ridge: 1
Best Alpha for Lasso: 100


### 12. Compare Ridge and Lasso based on:

- Training error

In [42]:
best_ridge = ridge_cv.best_estimator_
best_lasso = lasso_cv.best_estimator_

ridge_train_pred = best_ridge.predict(X_train_scaled)
lasso_train_pred = best_lasso.predict(X_train_scaled)

print("Ridge Training Error:",mean_squared_error(y_train, ridge_train_pred))
print("Lasso Training Error:",mean_squared_error(y_train, lasso_train_pred))

Ridge Training Error: 6162302702017.451
Lasso Training Error: 6162283933770.826


- validation error

In [43]:
ridge_test_pred = best_ridge.predict(X_test_scaled)
lasso_test_pred = best_lasso.predict(X_test_scaled)

print("Ridge Validation Error:",mean_squared_error(y_test, ridge_test_pred))
print("Lasso Validation Error:", mean_squared_error(y_test, lasso_test_pred))

Ridge Validation Error: 6451358176619.913
Lasso Validation Error: 6450219281451.082


- Feature coefficient behaviour

In [44]:
coef = pd.DataFrame({
    'Feature': X.columns,
    'Ridge Coefficient': best_ridge.coef_,
    'Lasso Coefficient': best_lasso.coef_
})

print(coef)

             Feature  Ridge Coefficient  Lasso Coefficient
0          area_sqft       6.948656e+06       6.955960e+06
1           bedrooms       2.952958e+05       2.891249e+05
2          bathrooms       2.730730e+05       2.731066e+05
3     location_score       3.678705e+06       3.680758e+06
4       property_age      -6.499425e+05      -6.500317e+05
5   distance_city_km      -2.852105e+04      -2.708265e+04
6        near_school       1.572028e+04       1.554959e+04
7         near_metro       5.254642e+04       5.253555e+04
8   crime_rate_index      -1.406921e+05      -1.405303e+05
9          sale_year       3.009622e+05       3.010515e+05
10        sale_month       1.742095e+04       1.728547e+04


## Part-D: Cross-Validation Strategies

### 13. Apply and compare the following cross-validation techniques:

- K-Fold Cross-Validation

In [45]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

ridge_scores = cross_val_score(best_ridge, X_train_scaled, y_train,cv=kf, scoring='neg_mean_squared_error')

lasso_scores = cross_val_score( best_lasso, X_train_scaled, y_train, cv=kf, scoring='neg_mean_squared_error')

ridge_mse = -np.mean(ridge_scores)
lasso_mse = -np.mean(lasso_scores)

print("Ridge K-Fold MSE:", ridge_mse)
print("Lasso K-Fold MSE:", lasso_mse)

Ridge K-Fold MSE: 6234200179333.174
Lasso K-Fold MSE: 6234224214947.506


- Stratified K-Fold Cross-Validation (by binning target variable)

In [46]:
y_bins = pd.qcut(y_train, q=5, labels=False)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

ridge_scores = cross_val_score(best_ridge, X_train_scaled, y_train,cv=skf.split(X_train_scaled, y_bins),scoring='neg_mean_squared_error')

lasso_scores = cross_val_score(best_lasso, X_train_scaled, y_train,cv=skf.split(X_train_scaled, y_bins),scoring='neg_mean_squared_error')

print("Ridge Stratified K-Fold MSE:", -np.mean(ridge_scores))
print("Lasso Stratified K-Fold MSE:", -np.mean(lasso_scores))

Ridge Stratified K-Fold MSE: 6210924866960.028
Lasso Stratified K-Fold MSE: 6210890104077.783


- Leave-One-Out Cross-Validation

In [47]:
loo = LeaveOneOut()

ridge_loo = cross_val_score(
    best_ridge,
    X_train_scaled,
    y_train,
    cv=loo,
    scoring='neg_mean_squared_error'
)

lasso_loo = cross_val_score(
    best_lasso,
    X_train_scaled,
    y_train,
    cv=loo,
    scoring='neg_mean_squared_error'
)

print("Ridge LOOCV MSE:", -np.mean(ridge_loo))
print("Lasso LOOCV MSE:", -np.mean(lasso_loo))

Ridge LOOCV MSE: 6215400744245.798
Lasso LOOCV MSE: 6215412674904.846


- Time Series Split (using a time-related feature)

In [48]:
df_time = df.sort_values('sale_date')

X_time = df_time.drop(columns=['property_id', 'sale_date', 'house_price_inr'])
y_time = df_time['house_price_inr']

scaler = StandardScaler()
X_time_scaled = scaler.fit_transform(X_time)

tscv = TimeSeriesSplit(n_splits=5)

ridge_scores = cross_val_score(best_ridge,X_time_scaled,y_time,cv=tscv,scoring='neg_mean_squared_error')

lasso_scores = cross_val_score(best_lasso,X_time_scaled,y_time,cv=tscv,scoring='neg_mean_squared_error')

print("Ridge Time Series MSE:", -np.mean(ridge_scores))
print("Lasso Time Series MSE:", -np.mean(lasso_scores))

Ridge Time Series MSE: 6276974299959.287
Lasso Time Series MSE: 6279360933831.891


### 14. Analyze how performance metrics vary across different CV strategies.

In [49]:
cv_results = pd.DataFrame({
    'CV Strategy': [
        'K-Fold',
        'Stratified K-Fold',
        'Leave-One-Out',
        'Time Series Split'
    ],
    'Ridge MSE': [
        ridge_mse,
        -np.mean(ridge_scores),      
        -np.mean(ridge_loo),
        -np.mean(ridge_scores)       
    ],
    'Lasso MSE': [
        lasso_mse,
        -np.mean(lasso_scores),      
        -np.mean(lasso_loo),
        -np.mean(lasso_scores)       
    ]
})

print(cv_results)

         CV Strategy     Ridge MSE     Lasso MSE
0             K-Fold  6.234200e+12  6.234224e+12
1  Stratified K-Fold  6.276974e+12  6.279361e+12
2      Leave-One-Out  6.215401e+12  6.215413e+12
3  Time Series Split  6.276974e+12  6.279361e+12


## Part-E: Tree-Based Regression Models

### 15. Implement Decision Tree Regression.

In [50]:
dt_model = DecisionTreeRegressor(random_state=42)

dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)

mse_dt = mean_squared_error(y_test, y_pred_dt)
r2_dt = r2_score(y_test, y_pred_dt)

print("Decision Tree MSE:", mse_dt)
print("Decision Tree R2 Score:", r2_dt)

Decision Tree MSE: 10826810122095.922
Decision Tree R2 Score: 0.8655641974314878


### 16. Control tree complexity using hyperparameters (max depth, min samples).

In [51]:
dt_tuned = DecisionTreeRegressor(max_depth=5,min_samples_split=10,min_samples_leaf=5,random_state=42)

dt_tuned.fit(X_train, y_train)

y_pred_tuned = dt_tuned.predict(X_test)

print("MSE:", mean_squared_error(y_test, y_pred_tuned))
print("R2 Score:", r2_score(y_test, y_pred_tuned))

MSE: 9385836459040.166
R2 Score: 0.8834566744111698


### 17. Implement Random Forest Regression.

In [52]:
rf_model = RandomForestRegressor(n_estimators=100,random_state=42)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest MSE:", mse_rf)
print("Random Forest R2 Score:", r2_rf)

Random Forest MSE: 5785958403336.219
Random Forest R2 Score: 0.9281561279076027


### 18. Compare single-tree vs ensemble performance.

In [53]:
dt_mse = mean_squared_error(y_test, y_pred_dt)
dt_r2 = r2_score(y_test, y_pred_dt)

rf_mse = mean_squared_error(y_test, y_pred_rf)
rf_r2 = r2_score(y_test, y_pred_rf)

comparison = pd.DataFrame({'Model': ['Decision Tree', 'Random Forest'],'MSE': [dt_mse, rf_mse],'R2 Score': [dt_r2, rf_r2]})

print(comparison)

           Model           MSE  R2 Score
0  Decision Tree  1.082681e+13  0.865564
1  Random Forest  5.785958e+12  0.928156


## Part-F: Support Vector Regression

### 19. Implement Support Vector Regression (SVR) with:

- Linear kernel

In [54]:
svr_linear = SVR(kernel='linear')

svr_linear.fit(X_train_scaled, y_train)

y_pred_svr_linear = svr_linear.predict(X_test_scaled)

mse_linear = mean_squared_error(y_test, y_pred_svr_linear)
r2_linear = r2_score(y_test, y_pred_svr_linear)

print("Linear SVR MSE:", mse_linear)
print("Linear SVR R2 Score:", r2_linear)

Linear SVR MSE: 80689212166031.83
Linear SVR R2 Score: -0.0019127401175407233


- Polynomial or RBF kernel

In [55]:
poly = SVR(kernel='poly')
poly.fit(X_train_scaled, y_train)
poly_pred = poly.predict(X_test_scaled)

rbf = SVR(kernel='rbf')
rbf.fit(X_train_scaled, y_train)
rbf_pred = rbf.predict(X_test_scaled)

print("Polynomial R2:", poly.score(X_test_scaled, y_test))
print("RBF R2:", rbf.score(X_test_scaled, y_test))

Polynomial R2: -0.0029191454904722303
RBF R2: -0.002946542549670861


### 20. Tune hyperparameters 

In [56]:
params = {'C': [1, 10, 100],'gamma': ['scale', 'auto'],'epsilon': [0.1, 0.5]}

grid = GridSearchCV(SVR(kernel='rbf'), params, cv=5)
grid.fit(X_train_scaled, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)

Best Parameters: {'C': 100, 'epsilon': 0.1, 'gamma': 'scale'}
Best Score: -0.006706270529206226


### 21. Compare SVR performance with linear and tree-based models.

In [57]:
print("Ridge R2:", best_ridge.score(X_test_scaled, y_test))
print("Lasso R2:", best_lasso.score(X_test_scaled, y_test))
print("SVR R2:", grid.score(X_test_scaled, y_test))
print("Decision Tree R2:", dt_model.score(X_test, y_test))
print("Random Forest R2:", rf_model.score(X_test, y_test))

Ridge R2: 0.9198939018648887
Lasso R2: 0.9199080434527005
SVR R2: -0.000627364427231214
Decision Tree R2: 0.8655641974314878
Random Forest R2: 0.9281561279076027


## Part-G: Model Comparison and Evaluation

### 22. Evaluate all models using appropriate regression metrics:

- MSE, MAE, RMSE

In [58]:
def evaluate(name, y_pred):
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    
    print(name, "-> MSE:", mse, "MAE:", mae, "RMSE:", rmse)

evaluate("Ridge", best_ridge.predict(X_test_scaled))
evaluate("Lasso", best_lasso.predict(X_test_scaled))
evaluate("Decision Tree", y_pred_dt)
evaluate("Random Forest", y_pred_rf)
evaluate("SVR", grid.predict(X_test_scaled))

Ridge -> MSE: 6451358176619.913 MAE: 1945385.3977088502 RMSE: 2539952.396526343
Lasso -> MSE: 6450219281451.082 MAE: 1945525.4816209406 RMSE: 2539728.190466665
Decision Tree -> MSE: 10826810122095.922 MAE: 2395187.719736842 RMSE: 3290411.8468811656
Random Forest -> MSE: 5785958403336.219 MAE: 1757755.0067894738 RMSE: 2405401.9213711913
SVR -> MSE: 80585694217177.05 MAE: 6985495.056820319 RMSE: 8976953.504233886


- R-2 Score

In [59]:
print("Ridge R2:", r2_score(y_test, best_ridge.predict(X_test_scaled)))
print("Lasso R2:", r2_score(y_test, best_lasso.predict(X_test_scaled)))
print("Decision Tree R2:", r2_score(y_test, y_pred_dt))
print("Random Forest R2:", r2_score(y_test, y_pred_rf))
print("SVR R2:", r2_score(y_test, grid.predict(X_test_scaled)))

Ridge R2: 0.9198939018648887
Lasso R2: 0.9199080434527005
Decision Tree R2: 0.8655641974314878
Random Forest R2: 0.9281561279076027
SVR R2: -0.000627364427231214


### 23. Compare: 

- Regularized Linear Models

In [60]:
print("Ridge R2:", best_ridge.score(X_test_scaled, y_test))
print("Lasso R2:", best_lasso.score(X_test_scaled, y_test))

Ridge R2: 0.9198939018648887
Lasso R2: 0.9199080434527005


- Tree-Based Models

In [61]:
print("Decision Tree R2:", dt_model.score(X_test, y_test))
print("Random Forest R2:", rf_model.score(X_test, y_test))

Decision Tree R2: 0.8655641974314878
Random Forest R2: 0.9281561279076027


- Support Vector Regression

In [62]:
print("Linear SVR R2:", svr_linear.score(X_test_scaled, y_test))
print("Polynomial SVR R2:", poly.score(X_test_scaled, y_test))
print("RBF SVR R2:", rbf.score(X_test_scaled, y_test))

Linear SVR R2: -0.0019127401175407233
Polynomial SVR R2: -0.0029191454904722303
RBF SVR R2: -0.002946542549670861


### 24. Identify signs of overfitting or underfitting in each model.

In [63]:
print("Decision Tree:")
print("Train:", dt_model.score(X_train, y_train))
print("Test :", dt_model.score(X_test, y_test))

print("\nRandom Forest:")
print("Train:", rf_model.score(X_train, y_train))
print("Test :", rf_model.score(X_test, y_test))

print("\nSVR:")
print("Train:", grid.score(X_train_scaled, y_train))
print("Test :", grid.score(X_test_scaled, y_test))

Decision Tree:
Train: 1.0
Test : 0.8655641974314878

Random Forest:


Train: 0.9893457381875551
Test : 0.9281561279076027

SVR:
Train: -0.004414311843415231
Test : -0.000627364427231214


## Part-H: Final Analysis and Reporting

### 25. Prepare a final report covering:

( i ) Best-performing model and justification
- Best-performing model: 
    - The model with the lowest RMSE and highest R² score is considered the best-performing model.
    - Its predicted values are closer to the actual values.
    - It gives better accuracy on unseen test data.
    - Therefore, it performs better compared to the other models.

- Justification: 
    - A lower RMSE indicates that the model has fewer prediction errors.
    - A higher R² score means the model explains more variation in the data.
    - These metrics help us compare the performance of different models.
    - Hence, the model with better RMSE and R² values is more reliable.

( ii ) Impact of regularization
- Regularization helps to reduce overfitting by adding a penalty to large model coefficients.
It makes the model simpler and improves its performance on unseen data.
It can slightly increase training error but usually improves test accuracy and generalization.

( iii ) Role of cross-validation in model stability
- Cross-validation tests the model on different subsets of the dataset.
- It helps check whether the model gives consistent performance across different data samples.
- It reduces the risk of overfitting and improves model reliability.
- cross-validation helps ensure the model is stable and generalizes well to unseen data

( iv ) Comparison of linear vs non-linear regressors
- Linear Regressors:
    - Linear regressors assume a straight-line relationship between input and output variables.
    - They are simple, fast, and easy to interpret.
    - They work well when the relationship in the data is approximately linear.

- Non-Linear Regressors:
    - Non-linear regressors can capture complex and curved relationships in the data.
    - They may provide better accuracy when the dataset contains non-linear patterns.
    - They are usually more complex and may have a higher risk of overfitting.

( v ) Business interpratation of results
- The model results help businesses understand patterns and predict future outcomes from available data.
- Accurate predictions can support better decisions related to planning, pricing, sales, and resource allocation.
- The results can also help identify important factors that have the greatest impact on business performance.
- The model helps reduce risk and improve decision-making.